# 🎬 RL Movie - ML-Agents 自動トレーニング

**使い方: 上から順にセルを実行するだけ！**

### 事前準備
Unity の `RLMovie > Build for Colab` メニューで ZIP を作成し、
Google Drive の `RL-Movie/Builds/` にアップロードしてください。

---
## 📋 Step 0: 設定（ここだけ手動で変更）

In [ ]:
#@title ⚙️ トレーニング設定 { run: "auto" }

# === ここを変更 ===
SCENARIO_NAME = "RollerBall"  #@param {type:"string"}
RUN_ID = "run_001"  #@param {type:"string"}
MAX_STEPS = 500000  #@param {type:"integer"}
RESUME_TRAINING = False  #@param {type:"boolean"}

# === 自動設定（変更不要） ===
DRIVE_PROJECT = "/content/drive/MyDrive/RL-Movie"
DRIVE_BUILDS = f"{DRIVE_PROJECT}/Builds"
DRIVE_RESULTS = f"{DRIVE_PROJECT}/Results"
DRIVE_MODELS = f"{DRIVE_PROJECT}/Models"
WORK_DIR = "/content/training"
BUILD_DIR = f"{WORK_DIR}/{SCENARIO_NAME}"
CONFIG_DIR = f"{BUILD_DIR}/Config"

print(f"🎯 Scenario: {SCENARIO_NAME}")
print(f"🏷️ Run ID:   {RUN_ID}")
print(f"📊 Max Steps: {MAX_STEPS:,}")
print(f"🔄 Resume:   {RESUME_TRAINING}")

---
## 🔧 Step 1: 環境セットアップ（初回のみ数分かかります）

In [ ]:
#@title 📦 ML-Agents インストール { run: "auto" }
%%capture install_log

!pip install mlagents==1.1.0
!pip install protobuf==3.20.3

# バージョン確認
import subprocess
result = subprocess.run(['mlagents-learn', '--version'], capture_output=True, text=True)
print(f"✅ ML-Agents installed: {result.stdout.strip()}")

In [ ]:
#@title 📂 Google Drive マウント { run: "auto" }
from google.colab import drive
import os

drive.mount('/content/drive')

# 必要なフォルダを作成
for d in [DRIVE_BUILDS, DRIVE_RESULTS, DRIVE_MODELS]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Drive mounted")
print(f"📁 Builds:  {DRIVE_BUILDS}")
print(f"📁 Results: {DRIVE_RESULTS}")
print(f"📁 Models:  {DRIVE_MODELS}")

---
## 📦 Step 2: ビルドの展開

In [ ]:
#@title 📤 ZIP を展開して準備 { run: "auto" }
import zipfile
import glob
import shutil

# ZIP ファイルを検索
zip_path = f"{DRIVE_BUILDS}/{SCENARIO_NAME}.zip"

if not os.path.exists(zip_path):
    # 名前が違う場合、利用可能な ZIP を一覧
    available = glob.glob(f"{DRIVE_BUILDS}/*.zip")
    print(f"❌ {SCENARIO_NAME}.zip が見つかりません！")
    print(f"\n📁 利用可能な ZIP:")
    for z in available:
        size_mb = os.path.getsize(z) / (1024*1024)
        print(f"  - {os.path.basename(z)} ({size_mb:.1f} MB)")
    if not available:
        print("  (なし - Unity で Build for Colab を実行してアップロードしてください)")
    raise FileNotFoundError(f"{zip_path} not found")

# 作業ディレクトリをクリーンアップ
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# ZIP 展開
print(f"📦 Extracting {SCENARIO_NAME}.zip ...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(WORK_DIR)

# Linux 実行ファイルに権限を付与
exec_files = glob.glob(f"{WORK_DIR}/**/*.x86_64", recursive=True)
for f in exec_files:
    os.chmod(f, 0o755)
    print(f"✅ Executable: {f}")

# 設定ファイルを確認
config_files = glob.glob(f"{WORK_DIR}/**/Config/*.yaml", recursive=True)
print(f"\n📋 Config files:")
for c in config_files:
    print(f"  - {c}")

if not exec_files:
    print("\n❌ 実行ファイルが見つかりません。ZIP の中身を確認してください。")
    !ls -la {WORK_DIR}
    raise FileNotFoundError("No .x86_64 executable found")

---
## 🧠 Step 3: トレーニング実行

In [ ]:
#@title ⚡ Max Steps をコンフィグに反映 { run: "auto" }
import yaml

# 設定ファイルを読み込んで max_steps を更新
if config_files:
    config_path = config_files[0]
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # max_steps を更新
    for behavior_name, behavior_config in config.get('behaviors', {}).items():
        behavior_config['max_steps'] = MAX_STEPS
        print(f"📊 {behavior_name}: max_steps = {MAX_STEPS:,}")
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"✅ Config updated: {config_path}")
else:
    print("⚠️ Config file not found, using defaults")
    config_path = None

In [ ]:
#@title 🚀 トレーニング開始 { run: "auto" }

exec_path = exec_files[0]
resume_flag = "--resume" if RESUME_TRAINING else "--force"

cmd = f"""mlagents-learn {config_path} \
    --env={exec_path} \
    --run-id={RUN_ID} \
    --no-graphics \
    --results-dir={DRIVE_RESULTS} \
    {resume_flag}"""

print("🧠 Training command:")
print(cmd)
print("\n" + "="*60)
print("🎓 トレーニングを開始します...")
print("  完了まで数分〜数十分かかります。")
print("  結果は Google Drive に自動保存されます。")
print("="*60 + "\n")

!{cmd}

---
## 📊 Step 4: 結果確認

In [ ]:
#@title 📈 TensorBoard で学習曲線を確認 { run: "auto" }
%load_ext tensorboard
%tensorboard --logdir {DRIVE_RESULTS}

In [ ]:
#@title 🧠 学習済みモデルの確認・コピー { run: "auto" }
import glob
import shutil

# ONNX モデルを検索
models = glob.glob(f"{DRIVE_RESULTS}/{RUN_ID}/**/*.onnx", recursive=True)

if models:
    print("🧠 学習済みモデル:")
    for m in models:
        size_kb = os.path.getsize(m) / 1024
        print(f"  ✅ {m} ({size_kb:.1f} KB)")
    
    # Models フォルダにコピー
    os.makedirs(DRIVE_MODELS, exist_ok=True)
    for m in models:
        model_name = os.path.basename(m)
        dest = f"{DRIVE_MODELS}/{SCENARIO_NAME}_{RUN_ID}_{model_name}"
        shutil.copy2(m, dest)
        print(f"\n📥 コピー先: {dest}")
    
    print(f"\n" + "="*60)
    print(f"🎬 次のステップ:")
    print(f"  1. Google Drive の RL-Movie/Models/ からモデルをダウンロード")
    print(f"  2. Unity の Assets/StreamingAssets/ に配置")
    print(f"  3. エージェントの Behavior Parameters > Model に設定")
    print(f"  4. Behavior Type を 'Inference Only' に変更")
    print(f"  5. Play して動画を録画！ 🎥")
    print("="*60)
else:
    print("⚠️ モデルが見つかりません。トレーニングが完了していない可能性があります。")
    print(f"\n📁 Results directory contents:")
    !ls -la {DRIVE_RESULTS}/{RUN_ID}/ 2>/dev/null || echo "Run directory not found"

In [ ]:
#@title 📥 モデルを直接ダウンロード（オプション） { run: "auto" }
from google.colab import files

if models:
    latest = models[-1]
    print(f"📥 Downloading: {os.path.basename(latest)}")
    files.download(latest)
else:
    print("⚠️ ダウンロードできるモデルがありません。")